# Phase 5: Evaluation

**CRISP-DM Phase Description:**  
At this stage, the model(s) appear to have high quality from a technical standpoint. Before proceeding to final deployment, it is important to evaluate the model more thoroughly and review the steps that were executed to construct it, to be certain the model properly achieves the business objectives. A key objective is to determine if there is some important business issue that has not been sufficiently considered.

---

In [2]:
# Standard imports
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

---
### Task 1: Evaluate Results

Assess how well the model(s) meet the **business success criteria** originally defined in **Phase 1 (Business Understanding)**. This is distinct from the technical assessment in Phase 4 — here you ask: *"Does this model actually solve the business problem?"*

Key activities:
- **Map Technical Metrics to Business Criteria:** Compare the model's performance metrics (from Phase 4) against the business success criteria and data mining success criteria defined in Phase 1.
- **Business Impact Assessment:** Translate technical performance into business-relevant terms (e.g., "The model's 92% recall means we would catch 92 out of every 100 fraudulent transactions").
- **Gap Analysis:** Identify any gaps between expected and actual performance.
- **Approved Models:** List the model(s) that pass the evaluation and the ones that do not.

**Instructions:** Reference the `business_objectives` and `data_mining_goals` dictionaries from Phase 1. Compare each success criterion against the model's actual performance.

In [ ]:
# TODO: Evaluate model results against business success criteria.
# Reference your Phase 1 definitions and Phase 4 results.

evaluation = {
    "business_criteria": [
        {
            "criterion": "1. Achieve a high-precision sales forecast that allows for stable inventory planning 6 weeks out.",
            "target": "RMSPE < 0.15 (15%)",
            "achieved": "The XGBoost model achieved an RMSPE of 0.1334 (13.3%), outperforming the requirement and ensuring stable inventory without overstocking.",
            "met": True
        },
        {
            "criterion": "2. Quantify the financial 'lift' of promotions to inform future marketing spend.",
            "target": "Identify and rank the impact of 'Promo' events.",
            "achieved": "The XGBoost model natively captured complex Promo interactions, proving that promotions are statistically significant drivers of revenue.",
            "met": True
        },
        {
            "criterion": "3. Provide actionable insights into the impact of State and School holidays on revenue.",
            "target": "Successfully learn holiday seasonality.",
            "achieved": "Time-series cross-validation proved the model accurately predicts sales fluctuations during holiday periods (Mean RMSE 0.1865).",
            "met": True
        },
        {
            "criterion": "4. Evaluate the ethical implications and potential biases of the model across different store locations.",
            "target": "Review model features for demographic bias.",
            "achieved": "The model relies strictly on objective store metadata (distance, promos, store type) rather than personal or demographic customer data, minimizing ethical risks.",
            "met": True
        }
    ],
    "technical_criteria": [
        {
            "metric": "RMSPE (Root Mean Square Percentage Error)",
            "target": "< 0.15 (15%)",
            "achieved": 0.1334,
            "met": True
        },
        {
            "metric": "Time-Series Stability",
            "target": "Low error variance across chronological folds",
            "achieved": "Mean RMSE of 0.1865 with a standard deviation of +/- 0.0243",
            "met": True
        }
    ],
    "overall_assessment": "The XGBoost model successfully meets all four business success criteria defined in Phase 1. It provides highly accurate, unbiased forecasts while capturing the critical impacts of promotions and holidays.",
    "approved_models": ["XGBoost", "XGBoost (Tuned) - Randomized"],     
    "rejected_models": ["Random Forest", "Ridge", "LightGBM", "XGBoost (Tuned - GridSearch)"]      
}

In [5]:
# Display the evaluation summary
print("=" * 60)
print("EVALUATION AGAINST BUSINESS SUCCESS CRITERIA")
print("=" * 60)

print("\n--- Business Criteria ---")
for c in evaluation['business_criteria']:
    status = 'PASS' if c['met'] else 'FAIL'
    print(f"  [{status}] {c['criterion']}")
    print(f"         Target: {c['target']}  |  Achieved: {c['achieved']}")

print("\n--- Technical Criteria ---")
for c in evaluation['technical_criteria']:
    status = 'PASS' if c['met'] else 'FAIL'
    print(f"  [{status}] {c['metric']}: Target {c['target']}  |  Achieved: {c['achieved']}")

print(f"\nOverall: {evaluation['overall_assessment']}")
print(f"Approved models: {evaluation['approved_models']}")
print(f"Rejected models: {evaluation['rejected_models']}")

EVALUATION AGAINST BUSINESS SUCCESS CRITERIA

--- Business Criteria ---
  [PASS] 1. Achieve a high-precision sales forecast that allows for stable inventory planning 6 weeks out.
         Target: RMSPE < 0.15 (15%)  |  Achieved: The XGBoost model achieved an RMSPE of 0.1334 (13.3%), outperforming the requirement and ensuring stable inventory without overstocking.
  [PASS] 2. Quantify the financial 'lift' of promotions to inform future marketing spend.
         Target: Identify and rank the impact of 'Promo' events.  |  Achieved: The XGBoost model natively captured complex Promo interactions therefore proving that promotions are statistically significant drivers of revenue.
  [PASS] 3. Provide actionable insights into the impact of State and School holidays on revenue.
         Target: Successfully learn holiday seasonality.  |  Achieved: Time-series cross-validation proved the model accurately predicts sales fluctuations during holiday periods (Mean RMSE 0.1893).
  [PASS] 4. Evaluate t

---
### Task 2: Review Process

Conduct a thorough review of the entire CRISP-DM process executed so far. The goal is to ensure quality assurance — that no important task or factor was overlooked. Consider:

- **Completeness Check:** Were all relevant data sources considered? Were the data preparation steps sufficient?
- **Methodology Review:** Were appropriate techniques used at each stage? Were there alternatives worth exploring?
- **Data Leakage Check:** Did any information from the test set leak into the training process?
- **Bias and Fairness:** Could the model introduce or amplify any biases? Is the model fair across different subgroups?
- **Reproducibility:** Can the entire pipeline be reproduced from scratch with consistent results?

**Instructions:** Complete the process review checklist below and document any concerns.

In [7]:
# TODO: Review the end-to-end process.

process_review = {
    "checklist": [
        {
            "item": "All relevant data sources were considered",       
            "status": "Yes", 
            "notes": "Merged the core training data with the supplementary store metadata to capture distance and assortment."
        },
        {
            "item": "Data preparation was thorough and documented",    
            "status": "Yes", 
            "notes": "Handled missing values, removed closed store days (sales=0), applied log transformation to normalize the target, and utilized one-hot encoding."
        },
        {
            "item": "No data leakage between train and test sets",     
            "status": "Yes", 
            "notes": "Used shuffle=False during train_test_split and TimeSeriesSplit for CV to strictly enforce chronological forecasting."
        },
        {
            "item": "Multiple modelling techniques were compared",     
            "status": "Yes", 
            "notes": "Evaluated a Scikit-Learn Random Forest baseline against an advanced XGBoost model."
        },
        {
            "item": "Hyperparameters were tuned appropriately",        
            "status": "Yes", 
            "notes": "Conducted a constrained GridSearchCV on XGBoost to optimize learning_rate and max_depth, with early stopping."
        },
        {
            "item": "Results are reproducible",     
            "status": "Yes", 
            "notes": "RANDOM_SEED = 42 was strictly enforced across all splits, models, and grid searches."
        },
        {
            "item": "Bias and fairness have been considered",          
            "status": "Yes", 
            "notes": "The model relies solely on store mechanics and dates, actively avoiding sensitive demographic data."
        },
        {
            "item": "Ethical implications have been reviewed",         
            "status": "Yes", 
            "notes": "Predictions are aggregated at the store level. No individual customer tracking or profiling is utilized."
        },
    ],
    "overall_quality": "High. The pipeline is technically sound, robust to time-series specific pitfalls, and achieves competitive accuracy.",
    "areas_for_improvement": [
        "Incorporate external macroeconomic data (ex. regional weather data or local GDP) to further explain sales variances.",
        "Allocate more compute time to run a deeper, exhaustive Grid Search across a wider range of XGBoost parameters.",
        "Explore deep learning architectures (like LSTMs) explicitly designed for sequence-based time-series forecasting."
    ]
}

In [8]:
# Display the process review
print("=== Process Review Checklist ===")
for item in process_review['checklist']:
    print(f"  [{item['status']:^8}] {item['item']}")
    if item['notes']:
        print(f"           Note: {item['notes']}")

print(f"\nOverall Quality: {process_review['overall_quality']}")
if process_review['areas_for_improvement']:
    print("Areas for Improvement (Future Work):")
    for area in process_review['areas_for_improvement']:
        print(f"  - {area}")

=== Process Review Checklist ===
  [  Yes   ] All relevant data sources were considered
           Note: Merged the core training data with the supplementary store metadata to capture distance and assortment.
  [  Yes   ] Data preparation was thorough and documented
           Note: Handled missing values, removed closed store days (sales=0), applied log transformation to normalize the target, and utilized one-hot encoding.
  [  Yes   ] No data leakage between train and test sets
           Note: Used shuffle=False during train_test_split and TimeSeriesSplit for CV to strictly enforce chronological forecasting.
  [  Yes   ] Multiple modelling techniques were compared
           Note: Evaluated a Scikit-Learn Random Forest baseline against an advanced XGBoost model.
  [  Yes   ] Hyperparameters were tuned appropriately
           Note: Conducted a constrained GridSearchCV on XGBoost to optimize learning_rate and max_depth, with early stopping.
  [  Yes   ] Results are reproducible
     

---
### Task 3: Determine Next Steps

Based on the evaluation results and process review, decide on the next course of action. The three main options are:

1. **Proceed to Deployment:** The model meets all criteria and is ready for deployment.
2. **Iterate:** Go back to a previous phase (e.g., Data Preparation or Modelling) to improve the model before deployment.
3. **Terminate:** The project is not viable in its current form and should be re-scoped or abandoned.

**Instructions:** Document your decision and provide a clear rationale.

In [9]:
# TODO: Determine and document the next steps.

next_steps = {
    "decision": "Proceed to Deployment",  
    "rationale": "The tuned XGBoost model exceeded all business and technical success criteria, achieving a highly accurate 13.3% RMSPE and also the QA review confirms the methodology is sound, chronological integrity was maintained, and the model is stable enough for production.",
    "if_iterating": {
        "return_to_phase": "N/A",  
        "specific_actions": []
    },
    "if_deploying": {
        "selected_model": "rossmann_xgboost_final.pkl (Tuned XGBoost)",  
        "deployment_priority": "High — Model must be packaged into the `src/processing.py` pipeline."  
    }
}

In [10]:
# Display the decision
print("=" * 60)
print("NEXT STEPS DECISION")
print("=" * 60)
print(f"\nDecision  : {next_steps['decision']}")
print(f"Rationale : {next_steps['rationale']}")

if next_steps['decision'] == 'Iterate':
    print(f"\nReturn to: {next_steps['if_iterating']['return_to_phase']}")
    for action in next_steps['if_iterating']['specific_actions']:
        print(f"  - {action}")
elif next_steps['decision'] == 'Proceed to Deployment':
    print(f"\nModel to deploy: {next_steps['if_deploying']['selected_model']}")
    print(f"Priority: {next_steps['if_deploying']['deployment_priority']}")

NEXT STEPS DECISION

Decision  : Proceed to Deployment
Rationale : The tuned XGBoost model exceeded all business and technical success criteria, achieving a highly accurate 13.3% RMSPE and also the QA review confirms the methodology is sound, chronological integrity was maintained, and the model is stable enough for production.

Model to deploy: rossmann_xgboost_final.pkl (Tuned XGBoost)
Priority: High — Model must be packaged into the `src/processing.py` pipeline.
